# 01 — Initial Exploration
**Phase 1, Step 7** · First analytical pass on the raw Adzuna dataset.

**Purpose**: discover what's in the data and — just as importantly — what's *broken* in it. Every anomaly found here becomes a Phase 2 cleaning task. This notebook stays deliberately shallow: no normalization, no skill extraction, no conclusions that the data quality can't yet support.

**Prerequisites**: Postgres running, migrations applied, ingestion completed (real or `scripts/seed_demo_data.py`).

In [1]:
import os

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
engine = create_engine(os.environ["DATABASE_URL"])

pd.set_option("display.max_colwidth", 120)


def q(sql: str) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


q("select count(*) as jobs from jobs")

,jobs
0,600


## Section 1 — Volume & freshness

How much data do we have, and is it fresh enough to trust? A dataset dominated by stale postings would bias every downstream salary and demand analysis.

In [2]:
overview = q("""
    select
        count(*)                                        as total_jobs,
        count(distinct company_id)                      as distinct_companies,
        count(distinct location_id)                     as distinct_locations,
        min(posted_at)::date                            as oldest_posting,
        max(posted_at)::date                            as newest_posting,
        count(*) filter (where posted_at is null)       as missing_posted_at
    from jobs
""")
overview

,total_jobs,distinct_companies,distinct_locations,oldest_posting,newest_posting,missing_posted_at
0,600,15,9,2026-06-01,2026-06-28,0


In [3]:
by_day = q("""
    select posted_at::date as posted_date, count(*) as postings
    from jobs
    where posted_at is not null
    group by 1
    order by 1
""")
fig = px.bar(
    by_day, x="posted_date", y="postings", title="Postings by posted date — freshness check"
)
fig.update_layout(xaxis_title=None, yaxis_title="postings")
fig.show()

## Section 2 — Top companies

Who posts the most? **Expect this chart to dramatically expose the company-name-normalization problem** — the same employer appearing under multiple strings, and staffing agencies (TekSystems, Robert Half, Insight Global) masking the true hiring company. Do *not* fix anything here; just document the damage for Phase 2.

In [4]:
top_companies = q("""
    select c.canonical_name as company, count(*) as postings
    from jobs j
    join companies c on c.id = j.company_id
    group by 1
    order by postings desc
    limit 20
""")
fig = px.bar(
    top_companies.sort_values("postings"),
    x="postings",
    y="company",
    orientation="h",
    title="Top 20 companies by posting count (raw, unnormalized names)",
)
fig.update_layout(yaxis_title=None)
fig.show()

In [5]:
# How many jobs have no company at all?
q("""
    select
        count(*) filter (where company_id is null) as jobs_without_company,
        round(100.0 * count(*) filter (where company_id is null) / count(*), 1) as pct
    from jobs
""")

,jobs_without_company,pct
0,48,8.0


## Section 3 — Salary availability

What fraction of postings disclose salary, and can those figures be trusted? Two known traps:
1. **Adzuna estimates**: `salary_is_predicted = true` means the number is Adzuna's model output, not the employer's disclosure — these must be separable in every future salary analysis.
2. **Period ambiguity**: Phase 1 records the period Adzuna implies (annual), but eyeball the low end of the distribution — values that look like hourly rates stored as annual are a classic contamination.

In [6]:
salary_coverage = q("""
    select
        count(*)                                                    as total,
        count(salary_min)                                           as with_salary,
        round(100.0 * count(salary_min) / count(*), 1)              as pct_with_salary,
        count(*) filter (where salary_is_predicted)                 as adzuna_estimated,
        round(100.0 * count(*) filter (where salary_is_predicted)
              / nullif(count(salary_min), 0), 1)                    as pct_estimated_of_disclosed
    from jobs
""")
salary_coverage

,total,with_salary,pct_with_salary,adzuna_estimated,pct_estimated_of_disclosed
0,600,294,49.0,138,46.9


In [7]:
salaries = q("""
    select salary_min, salary_max, salary_is_predicted
    from jobs
    where salary_min is not null
""")
fig = px.histogram(
    salaries,
    x="salary_min",
    color="salary_is_predicted",
    nbins=40,
    title="salary_min distribution — split by whether Adzuna estimated it",
    labels={"salary_is_predicted": "Adzuna estimate"},
)
fig.show()

# Suspiciously low values -> probable hourly-as-annual contamination (Phase 2 task)
suspicious = salaries[salaries["salary_min"] < 25000]
print(f"Postings with salary_min < $25k (possible hourly figures): {len(suspicious)}")

Postings with salary_min < $25k (possible hourly figures): 0


## Section 4 — Geography

Where is the demand? Phase 1's remote signal is deliberately crude (the word "remote" in title/location display name), so treat the remote fraction as a **floor**, not an estimate.

In [8]:
top_locations = q("""
    select
        coalesce(l.city, l.state, l.country, '(unknown)') as location,
        count(*) as postings
    from jobs j
    left join locations l on l.id = j.location_id
    group by 1
    order by postings desc
    limit 15
""")
fig = px.bar(
    top_locations.sort_values("postings"),
    x="postings",
    y="location",
    orientation="h",
    title="Top locations by posting count",
)
fig.update_layout(yaxis_title=None)
fig.show()

In [9]:
q("""
    select
        count(*) filter (where is_remote)                        as remote_flagged,
        round(100.0 * count(*) filter (where is_remote) / count(*), 1) as pct_remote_floor,
        count(*) filter (where location_id is null)              as no_location
    from jobs
""")

,remote_flagged,pct_remote_floor,no_location
0,18,3.0,0


## Section 5 — Description sniff test

Before building rigorous skill extraction (Phase 2, spaCy), get a feel for the raw material: what do descriptions actually look like, and how often do the obvious skills get mentioned with a naive `ILIKE`? These numbers are noisy by construction (e.g. `ILIKE '%r%'` would be useless for R — note how it's word-bounded below) and exist only to calibrate expectations.

In [10]:
sample = q(
    "select title, left(description, 300) as description_start from jobs order by random() limit 5"
)
for _, row in sample.iterrows():
    print(f"— {row['title']}\n  {row['description_start']}...\n")

— Analytics Engineer
  We are seeking a Analytics Engineer to join our team. Familiarity with dbt, Snowflake, and modern data stack tooling. This role involves stakeholder communication and data storytelling. Bachelor's degree in a quantitative field preferred......

— Data Analyst II
  We are seeking a Data Analyst II to join our team. You will build dashboards in Looker and write complex SQL queries. This role involves stakeholder communication and data storytelling. Bachelor's degree in a quantitative field preferred......

— Marketing Data Analyst
  We are seeking a Marketing Data Analyst to join our team. Familiarity with dbt, Snowflake, and modern data stack tooling. This role involves stakeholder communication and data storytelling. Bachelor's degree in a quantitative field preferred......

— Lead Data Analyst
  We are seeking a Lead Data Analyst to join our team. Familiarity with dbt, Snowflake, and modern data stack tooling. This role involves stakeholder communication and dat

In [11]:
skill_mentions = q("""
    select skill, count(*) as mentions,
           round(100.0 * count(*) / (select count(*) from jobs), 1) as pct_of_jobs
    from jobs,
    lateral (values
        ('SQL',      description ~* '\\msql\\M'),
        ('Python',   description ~* '\\mpython\\M'),
        ('Tableau',  description ~* '\\mtableau\\M'),
        ('Excel',    description ~* '\\mexcel\\M'),
        ('Power BI', description ~* '\\mpower bi\\M'),
        ('Looker',   description ~* '\\mlooker\\M'),
        ('R',        description ~* '\\mR\\M' and description ilike '%statist%')
    ) as s(skill, mentioned)
    where mentioned
    group by skill
    order by mentions desc
""")
fig = px.bar(
    skill_mentions,
    x="skill",
    y="pct_of_jobs",
    title="Naive skill mention rates (word-boundary regex — Phase 2 replaces this)",
)
fig.update_layout(yaxis_title="% of jobs mentioning")
fig.show()

## Open questions and data quality issues found

> **This cell is the actual deliverable of the notebook.** Every item here becomes a Phase 2 task. Update it after running against real data — the list below contains items already known from pipeline construction plus placeholders to verify.

1. **Company name duplication** — same employer under multiple raw strings (e.g. "Deloitte" vs "Deloitte Consulting LLP"). Quantify the top offenders against real data.
2. **Staffing-agency masking** — TekSystems / Robert Half / Insight Global style agencies post on behalf of unnamed clients; the "top companies" chart measures agency activity, not employer demand.
3. **Jobs with no company at all** — measure the fraction; decide Phase 2 handling (keep with NULL vs exclude from company-level analyses).
4. **Adzuna salary estimates** — `salary_is_predicted` rows must never be silently mixed with employer-disclosed salaries. Verify the split.
5. **Probable hourly-as-annual salary contamination** — check the `< $25k` bucket by hand against real data.
6. **Location hierarchy inconsistency** — Adzuna's `area` array varies in depth (country-only, state-only, full city); metro-area assignment in Phase 2 must handle all depths.
7. **Remote detection is a floor** — title/display-name keyword only; description-level signals ("work from home", "fully remote", hybrid language) are unhandled until Phase 2.
8. **Cross-query duplicate exposure** — the same posting can match multiple search queries; idempotent upserts dedupe it, but this means per-query counts don't sum to the jobs total. Document, don't "fix".
9. **`posted_at` trustworthiness** — Adzuna's `created` may reflect aggregation time, not original posting time; compare distributions against `first_seen_at` once 2+ weeks of snapshots exist.
10. **Title noise** — "Sr." vs "Senior", roman numerals ("Analyst II"), parenthetical modifiers ("(Hybrid)") — all fodder for Phase 2 seniority classification.